In [2]:
import os
import requests
import pandas as pd
from io import StringIO
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json
import geopandas as gpd
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
import torch.nn.functional as F
from torch.nn.utils import clip_grad_norm_
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, Subset
from sksurv.metrics import concordance_index_censored, integrated_brier_score
from scipy.stats import chi2
import random

# Visualization export configuration
from pathlib import Path

VISUALIZATION_DIR = Path("visualization_results")
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

def save_plotly_png(figure, filename, width=1600, height=900, scale=2):
    output_path = VISUALIZATION_DIR / filename
    try:
        figure.write_image(
            str(output_path),
            width=width,
            height=height,
            scale=scale,
        )
        print(f"Saved visualization: {output_path.resolve()}")
    except Exception as export_error:
        print(
            f"PNG export failed for {filename}: {export_error}\n"
            "Restart the kernel after installing kaleido, then rerun this cell."
        )


In [3]:
config = {
    "data": {
        "batch_size": 32768,
        "splits": [0.60, 0.20, 0.10, 0.10],
        "num_workers": 0
    },
    "training": {
        "epochs": 50,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "early_stopping_patience": 15,
        "gradient_clip_norm": 1.0,
        "checkpoint_dir": "./checkpoints",
        "checkpoint_name": "best.pt",
    },
    "evaluation": {
        "primary_metric": "c_index",
        "metrics_list": ["c_index", "ibs", "d_cal", "mae"]
    },
    "model": {
        "spatial_in": 12,
        "temporal_in": 1,
        "gcn_out": 64,
        "tcn_out": 32,
        "n_incident_features": 16,
    }
}

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make CUDA deterministic
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

In [5]:
!gdown 1LoLIWUP_nQ8rzpPZrkem8Q7yLtgCgyXh
!gdown 1Rpt18RdMyGbPu5jKSqtqmViu5XfVZH7c

Downloading...
From: https://drive.google.com/uc?id=1LoLIWUP_nQ8rzpPZrkem8Q7yLtgCgyXh
To: /content/Boundaries_-_Community_Areas_20260523.geojson
100% 2.08M/2.08M [00:00<00:00, 62.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Rpt18RdMyGbPu5jKSqtqmViu5XfVZH7c
From (redirected): https://drive.google.com/uc?id=1Rpt18RdMyGbPu5jKSqtqmViu5XfVZH7c&confirm=t&uuid=78a6e317-30b2-4146-baa3-8de94ce55b5c
To: /content/chicago_infra.csv
100% 109M/109M [00:01<00:00, 103MB/s]  


# EDA before Pre-processing

In [6]:
df = pd.read_csv("chicago_infra.csv")

# Calculate missing values percentage
total_rows = len(df)
missing_counts = df.isnull().sum().reset_index()
missing_counts.columns = ['Feature', 'Missing_Count']
missing_counts['Missing_Percentage'] = (missing_counts['Missing_Count'] / total_rows) * 100
missing_data = missing_counts[missing_counts['Missing_Count'] > 0]

missing_data

,Feature,Missing_Count,Missing_Percentage
3,closed_date,9271,1.426101
5,community_area,2950,0.453781
6,ward,2792,0.429476
7,latitude,1062,0.163361
8,longitude,1062,0.163361
10,parent_sr_number,544249,83.718508
14,electricity_grid,3569,0.548998
15,electrical_district,413376,63.587112


In [ ]:
# Visualize Missing Data
fig_missing = px.bar(
    missing_data,
    x='Feature',
    y='Missing_Percentage',
    title='Missing Values Percentage Before Preprocessing',
    text_auto='.2f',
    color='Missing_Percentage',
    color_continuous_scale='Reds'
)
fig_missing.update_layout(yaxis_title="Missing Percentage (%)", xaxis_title="Columns")
save_plotly_png(fig_missing, "01_missing_values_before_preprocessing.png", width=1600, height=900)
fig_missing.write_html("missing_values_before_preprocessing.html")
fig_missing.show()

In [42]:
fig_missing.write_html("missing_values_before_preprocessing.html")

In [ ]:
# Visualize Raw Ticket Status
fig_status = px.pie(
    df,
    names='status',
    title='Raw Ticket Status Distribution',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
save_plotly_png(fig_status, "02_raw_ticket_status_distribution.png", width=1400, height=900)
fig_status.show()

In [43]:
fig_status.write_html("raw_ticket_status_distribution.html")

# Preprocessing
Rows missing `community_area`, `latitude`, or `longitude` are dropped. Missing `closed_date` values are filled with the maximum extraction date to act as the right-censoring boundary. Corrupted rows generating negative `duration_days` are removed. Duplicate data with `duplicate` is TRUE being drop to prevent workload distortion.

In [9]:
# Drop rows lacking spatial assignment data
df_clean = df.dropna(subset=['community_area', 'latitude', 'longitude']).copy()

# Filter duplicate incidents to prevent workload distortion
if 'duplicate' in df_clean.columns:
    df_clean = df_clean[df_clean['duplicate'] == False]

# Convert timestamps
df_clean['created_date'] = pd.to_datetime(df_clean['created_date'])
df_clean['closed_date'] = pd.to_datetime(df_clean['closed_date'])

# Construct Survival Variables
extraction_date = df_clean['created_date'].max()
df_clean['end_date'] = df_clean['closed_date'].fillna(extraction_date)

df_clean['duration_days'] = (df_clean['end_date'] - df_clean['created_date']).dt.days
df_clean['event'] = (df_clean['status'] == 'Completed').astype(int)

# Filter corrupted negative durations
df_clean = df_clean[df_clean['duration_days'] >= 0]
df_clean = df_clean.drop(columns=['end_date'])

print(f"Total Rows after cleaning: {len(df_clean):,}")

Total Rows after cleaning: 541,625


# EDA After Preprocessing

In [10]:
distribution_stats = df_clean.groupby('sr_type')['duration_days'].describe()
distribution_stats

,count,mean,std,min,25%,50%,75%,max
sr_type,,,,,,,,
Pothole in Street Complaint,243487.0,22.428902,46.203912,0.0,0.0,5.0,19.0,1486.0
Street Light Out Complaint,188897.0,15.678322,76.548074,0.0,1.0,2.0,6.0,1597.0
Traffic Signal Out Complaint,48857.0,21.540741,95.543316,0.0,0.0,0.0,1.0,888.0
Tree Debris Clean-Up Request,60384.0,13.852030,21.813378,0.0,1.0,5.0,16.0,427.0


In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_clean,
    x='duration_days',
    hue='sr_type',
)
plt.title('Boxplot of Resolution Duration by Complaint Category', fontsize=14)
plt.xlabel('Duration (Days)', fontsize=12)
plt.ylabel('Category (sr_type)', fontsize=12)
plt.tight_layout()
plt.savefig(
    VISUALIZATION_DIR / "03_resolution_duration_boxplot.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
print(f"Saved visualization: {(VISUALIZATION_DIR / '03_resolution_duration_boxplot.png').resolve()}")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for category in df_clean['sr_type'].dropna().unique():
    subset = df_clean[df_clean['sr_type'] == category]
    sns.kdeplot(data=subset, x='duration_days', label=category, bw_adjust=1.5, fill=True, alpha=0.3)

plt.title('Density Distribution of Duration by Category (Capped at 60 Days)', fontsize=14)
plt.xlabel('Duration (Days)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.xlim(0, 60)
plt.legend(title='Category')
plt.tight_layout()
plt.savefig(
    VISUALIZATION_DIR / "04_resolution_duration_density.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
print(f"Saved visualization: {(VISUALIZATION_DIR / '04_resolution_duration_density.png').resolve()}")
plt.show()

In [44]:
df_clean.to_csv('clean_data.csv', index=False)

In [ ]:
# Spatial Point Map (Sub-sampled for browser rendering performance)
df_sample = df_clean.sample(n=min(5000, len(df_clean)), random_state=42)

with open('Boundaries_-_Community_Areas_20260523.geojson', 'r') as f:
    community_areas_geojson = json.load(f)

fig_map = px.scatter_map(
    df_sample,
    lat="latitude",
    lon="longitude",
    color="sr_type",
    title="Spatial Distribution of Infrastructure Complaints with Community Areas (5K Sample)",
    zoom=9.5,
    opacity=0.6
)

# Add the GeoJSON boundary layer and adjust the layout


# Keep the exported map tightly framed around Chicago community areas
chicago_map_bounds = {
    "west": -87.97,
    "east": -87.50,
    "south": 41.62,
    "north": 42.04,
}

fig_map.update_layout(
    map_center={"lat": 41.8372, "lon": -87.6860},
    map_zoom=9.72,
    margin={"r":10, "t":70, "l":10, "b":10},
    legend={
        "orientation": "h",
        "x": 0.5,
        "y": -0.02,
        "xanchor": "center",
        "yanchor": "top",
        "title_text": "Complaint category",
    },
    map_layers=[
        {
            "source": community_areas_geojson,
            "type": "line",
            "color": "black",
            "line": {"width": 1.0}
        }
    ]
)

save_plotly_png(fig_map, "05_spatial_distribution_infrastructure_complaints.png", width=1100, height=1200)
fig_map.show()

# Graph Convolution Network (GCN)

## Adjancecy Matrix

In [14]:
gdf = gpd.read_file('Boundaries_-_Community_Areas_20260523.geojson')

# Sort by Community Area Number
gdf['area_numbe'] = gdf['area_numbe'].astype(int)
gdf = gdf.sort_values('area_numbe').reset_index(drop=True)

# Project to Illinois State Plane East (EPSG:3435) for accurate distance math
print("Projecting CRS and calculating centroids...")
gdf_proj = gdf.to_crs(epsg=3435)

# Calculate the exact geometric centroid of each community area
gdf_proj['centroid'] = gdf_proj.geometry.centroid

n_nodes = len(gdf_proj)
A = np.zeros((n_nodes, n_nodes))

print("Computing spatial borders and inverse distances...")
for i in range(n_nodes):
    for j in range(n_nodes):
        if i != j:
            # Check if Area i and Area j share a physical geographic border
            if gdf_proj.geometry.iloc[i].touches(gdf_proj.geometry.iloc[j]):

                # Calculate straight-line distance between the two centroids
                dist = gdf_proj['centroid'].iloc[i].distance(gdf_proj['centroid'].iloc[j])

                # Assign inverse distance as the edge weight (closer = higher weight)
                A[i, j] = 10000.0 / dist

edges = np.where(A > 0)

# Create edge_index tensor: Shape [2, num_edges]
edge_index = torch.tensor(np.array([edges[0], edges[1]]), dtype=torch.long)

# Create edge_weight tensor: Shape [num_edges]
edge_weight = torch.tensor(A[edges], dtype=torch.float32)

print(f"Total Nodes: {n_nodes}")
print(f"Total Edges Found: {edge_index.shape[1]}")
print(f"edge_index shape: {list(edge_index.shape)}")
print(f"edge_weight shape: {list(edge_weight.shape)}")
print(f"Sample Weights: {edge_weight[:5].numpy()}")

Projecting CRS and calculating centroids...
Computing spatial borders and inverse distances...
Total Nodes: 77
Total Edges Found: 394
edge_index shape: [2, 394]
edge_weight shape: [394]
Sample Weights: [1.3581488  1.1698009  1.3581488  1.0168647  0.98513037]


## Model

In [15]:
class SpatialGCN(nn.Module):
    def __init__(self, in_features, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)

    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.silu(x)
        h_v = self.conv2(x, edge_index, edge_weight)
        return h_v

# Temporal Convolution Network (TCN)

In [16]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalTCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_dim, kernel_size=3, dilation=2):
        super().__init__()

        padding = (kernel_size - 1) * dilation
        self.network = nn.Sequential(
            # Layer 1
            nn.Conv1d(in_channels, hidden_channels, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.BatchNorm1d(hidden_channels),
            nn.SiLU(),

            # Layer 2 (Increase dilation to expand receptive field)
            nn.Conv1d(hidden_channels, hidden_channels, kernel_size, padding=padding*2, dilation=dilation*2),
            Chomp1d(padding*2),
            nn.BatchNorm1d(hidden_channels),
            nn.SiLU()
        )

        self.fc = nn.Linear(hidden_channels, out_dim)

    def forward(self, x):
        out = self.network(x)
        h_t = self.fc(out[:, :, -1])
        return h_t

# STGSurviNet

In [17]:
class STGSurviNet(nn.Module):
    def __init__(self, spatial_in, temporal_in, gcn_out, tcn_out, n_incident_features):
        super().__init__()
        self.spatial_extractor = SpatialGCN(in_features=spatial_in, hidden_dim=64, out_dim=gcn_out)
        self.temporal_extractor = TemporalTCN(in_channels=temporal_in, hidden_channels=32, out_dim=tcn_out)

        self.survival_layer = nn.Sequential(
            nn.Linear(gcn_out + tcn_out + n_incident_features, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_spatial, edge_index, edge_weight, x_temporal, node_indices=None, incident_features=None, ablation_mode=None):
        batch_size = x_spatial.shape[0]
        h_v = torch.zeros(batch_size, self.spatial_extractor.conv2.out_channels, device=x_spatial.device)
        h_t = torch.zeros(batch_size, self.temporal_extractor.fc.out_features, device=x_temporal.device)
        if ablation_mode != 'no_spatial':
            h_v = self.spatial_extractor(x_spatial, edge_index, edge_weight)
        if ablation_mode != 'no_temporal':
            h_t = self.temporal_extractor(x_temporal)

        Z_graph = torch.cat([h_v, h_t], dim=1)
        Z_batch = Z_graph[node_indices]
        if ablation_mode == 'no_incident':
            Z_fused = torch.cat([Z_batch, torch.zeros_like(incident_features)], dim=1)
        else:
            Z_fused = torch.cat([Z_batch, incident_features], dim=1)

        return self.survival_layer(Z_fused)

# Training

## Trainer

In [18]:
def d_calibration(event_times, predicted_sf_list, n_bins=10):
    """
    D-Calibration (Haider et al., 2020).
    Returns p-value: p > 0.05 = well calibrated.
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    counts = np.zeros(n_bins)

    for i, sf in enumerate(predicted_sf_list):
        t = event_times[i]
        prob = float(sf(t))  # predicted survival at actual event time
        prob = np.clip(prob, 0, 1)
        idx = np.searchsorted(bin_edges[1:], prob, side='right')
        idx = min(idx, n_bins - 1)
        counts[idx] += 1

    expected = len(event_times) / n_bins
    stat = np.sum((counts - expected) ** 2 / expected)
    p_value = 1 - chi2.cdf(stat, df=n_bins - 1)
    return {"statistic": stat, "p_value": p_value, "counts": counts}

In [19]:
class Trainer:
    def __init__(self, model, config, train_loader, val_loader, cal_loader, test_loader, edge_index, edge_weight, full_x_spatial, full_x_temporal):
        self.model        = model
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.cal_loader = cal_loader
        self.test_loader  = test_loader
        self.edge_index   = edge_index
        self.edge_weight  = edge_weight
        self.train_cfg    = config["training"]
        self.eval_cfg     = config["evaluation"]

        self.epochs        = self.train_cfg["epochs"]
        self.lr            = self.train_cfg["learning_rate"]
        self.weight_decay  = self.train_cfg["weight_decay"]
        self.patience      = self.train_cfg.get("early_stopping_patience", 15)
        self.ablation_mode = config.get("ablation_mode", None)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.full_x_spatial  = full_x_spatial.to(self.device)
        self.full_x_temporal = full_x_temporal.to(self.device)
        self.model.to(self.device)
        self.edge_index = self.edge_index.to(self.device)
        if self.edge_weight is not None:
            self.edge_weight = self.edge_weight.to(self.device)

        self.optimizer = torch.optim.AdamW([
            {'params': self.model.spatial_extractor.parameters(),  'lr': self.lr * 0.5},
            {'params': self.model.temporal_extractor.parameters(), 'lr': self.lr * 0.5},
            {'params': self.model.survival_layer.parameters(),     'lr': self.lr},
        ], weight_decay=self.weight_decay)

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='max',
            factor=0.5,
            patience=5,
            threshold=0.002,
            min_lr=1e-6,
        )

        self.best_c_index     = -float("inf")
        self.patience_counter = 0
        self.baselines = {}

        all_dur, all_evt = [], []
        for batch in train_loader:
            all_dur.append(batch["duration_days"])
            all_evt.append(batch["event"])
        dur = torch.cat(all_dur).numpy()
        evt = torch.cat(all_evt).numpy()
        self.y_train_structured = np.array(
            [(bool(e), t) for e, t in zip(evt, dur)],
            dtype=[("event", bool), ("time", float)],
        )

    def cox_loss(self, log_hazards, events, sample_size=512):
        log_hazards = torch.clamp(log_hazards, min=-10, max=10)
        exp_hazards   = torch.exp(log_hazards)
        log_risk_sums = torch.log(torch.cumsum(exp_hazards, dim=0))
        event_mask    = events.float()
        loss          = -torch.sum((log_hazards - log_risk_sums) * event_mask)
        return loss / torch.clamp(event_mask.sum(), min=1.0)

    def _forward(self, batch):
        node_indices      = batch["node_idx"].to(self.device)
        incident_features = batch["incident_features"].to(self.device)
        durations         = batch["duration_days"].to(self.device)
        events            = batch["event"].to(self.device)

        log_hazards = self.model(
            self.full_x_spatial, self.edge_index, self.edge_weight,
            self.full_x_temporal, node_indices=node_indices,
            incident_features=incident_features,
            ablation_mode=self.ablation_mode,
        ).squeeze(-1)

        return log_hazards, durations, events

    def _breslow_baseline(self, durations, events, risk_scores, sr_types=None):
        """
        If sr_types is None → single baseline (legacy).
        If sr_types provided → dict of {stype_int: (event_times, H0_cum)}.
        """
        def _fit_one(d, e, r):
            event_times     = np.sort(np.unique(d[e == 1]))
            sort_idx        = np.argsort(d)
            d_s, e_s, r_s   = d[sort_idx], e[sort_idx], r[sort_idx]
            rcr             = np.cumsum(r_s[::-1])[::-1]
            pos             = np.searchsorted(d_s, event_times, side='left')
            d_i             = np.array([np.sum(e_s[d_s == t]) for t in event_times])
            risk_set        = rcr[pos]
            h0              = np.where(risk_set > 0, d_i / risk_set, 0.0)
            return event_times, np.cumsum(h0)

        if sr_types is None:
            return {'all': _fit_one(durations, events, risk_scores)}

        baselines = {}
        for stype in np.unique(sr_types):
            mask = sr_types == stype
            if events[mask].sum() == 0:
                continue
            baselines[stype] = _fit_one(durations[mask], events[mask], risk_scores[mask])
        return baselines

    def _fit_breslow_on_cal(self):
        """Refit stratified Breslow on the held-out calibration set."""
        self.model.eval()
        all_hazards, all_durs, all_evts, all_types = [], [], [], []

        with torch.no_grad():
            for batch in self.cal_loader:
                node_indices      = batch["node_idx"].to(self.device)
                incident_features = batch["incident_features"].to(self.device)

                log_h = self.model(
                    self.full_x_spatial, self.edge_index, self.edge_weight,
                    self.full_x_temporal, node_indices=node_indices,
                    incident_features=incident_features,
                    ablation_mode=self.ablation_mode,
                ).squeeze(-1)

                all_hazards.append(log_h.cpu())
                all_durs.append(batch["duration_days"])
                all_evts.append(batch["event"])
                all_types.append(batch["sr_type_enc"])

        risks    = np.exp(torch.cat(all_hazards).numpy())
        durs     = torch.cat(all_durs).numpy()
        evts     = torch.cat(all_evts).numpy()
        sr_types = torch.cat(all_types).numpy()

        self.baselines = self._breslow_baseline(durs, evts, risks, sr_types=sr_types)

    def _predict_survival(self, risk_scores, query_times, breslow_times, H0_cum):
        H0_at_query = np.zeros(len(query_times))
        for j, t in enumerate(query_times):
            idx            = np.searchsorted(breslow_times, t, side="right") - 1
            H0_at_query[j] = H0_cum[idx] if idx >= 0 else 0.0
        return np.exp(-risk_scores[:, None] * H0_at_query[None, :])

    def train_epoch(self) -> float:
        self.model.train()
        total_loss = 0.0

        pbar = tqdm(self.train_loader, desc="Training")
        for batch in pbar:
            # Full forward pass — gradients now flow through GCN and TCN too
            log_hazards, durations, events = self._forward(batch)

            noisy_durations = durations + (torch.rand_like(durations) * 0.5)

            sort_idx    = torch.argsort(noisy_durations, descending=True)
            log_hazards = log_hazards[sort_idx]
            events      = events[sort_idx]

            self.optimizer.zero_grad()
            loss = self.cox_loss(log_hazards, events)
            if torch.isnan(loss):
                raise ValueError("NaN detected in loss — check input data or learning rate.")
            loss.backward()
            if self.train_cfg.get("gradient_clip_norm"):
                clip_grad_norm_(self.model.parameters(), self.train_cfg["gradient_clip_norm"])
            self.optimizer.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        return total_loss / len(self.train_loader)

    def evaluate(self, loader=None):
        if loader is None:
            loader = self.val_loader

        self.model.eval()
        all_hazards, all_durations, all_events, all_sr_types = [], [], [], []
        val_loss = 0.0

        with torch.no_grad():
            for batch in tqdm(loader, desc="Evaluating"):
                log_hazards, durations, events = self._forward(batch)
                sr_type_enc = batch["sr_type_enc"]

                sort_idx  = torch.argsort(durations, descending=True)
                val_loss += self.cox_loss(log_hazards[sort_idx], events[sort_idx]).item()

                all_hazards.append(log_hazards.cpu())
                all_durations.append(durations.cpu())
                all_events.append(events.cpu())
                all_sr_types.append(sr_type_enc)

        val_loss    /= len(loader)
        log_hazards  = torch.cat(all_hazards).numpy()
        durations    = torch.cat(all_durations).numpy()
        events       = torch.cat(all_events).numpy()
        sr_types     = torch.cat(all_sr_types).numpy()
        risk_scores  = np.exp(log_hazards)
        baselines    = self.baselines   # {stype_int: (bt, H0_cum)}

        y_structured = np.array(
            [(bool(e), t) for e, t in zip(events, durations)],
            dtype=[("event", bool), ("time", float)],
        )

        c_idx, *_ = concordance_index_censored(events.astype(bool), durations, risk_scores)

        # --- IBS: per-subject survival curves using their type's baseline ---
        times_grid = np.unique(np.percentile(durations, np.linspace(10, 90, 10)))
        t_min = max(bt.min() for bt, _ in baselines.values())
        t_max = min(bt.max() for bt, _ in baselines.values())
        times_grid = times_grid[(times_grid >= t_min) & (times_grid <= t_max)]

        surv_prob_matrix = np.zeros((len(risk_scores), len(times_grid)))
        for stype, (bt, h0) in baselines.items():
            mask = sr_types == stype
            if mask.sum() == 0:
                continue
            surv_prob_matrix[mask] = self._predict_survival(risk_scores[mask], times_grid, bt, h0)
        ibs = integrated_brier_score(self.y_train_structured, y_structured, surv_prob_matrix, times_grid)

        # --- D-Cal: stratified SF per event subject ---
        event_mask  = events == 1
        event_times = durations[event_mask]
        event_risks = risk_scores[event_mask]
        event_types = sr_types[event_mask]

        def make_sf(risk, stype):
            bt, h0 = baselines[stype]
            def sf(t):
                idx = np.searchsorted(bt, t, side="right") - 1
                return np.exp(-(h0[idx] if idx >= 0 else 0.0) * risk)
            return sf

        predicted_sf_list = [make_sf(r, s) for r, s in zip(event_risks, event_types)]
        d_cal = d_calibration(event_times, predicted_sf_list)

        # --- MAE: stratified median survival per event subject ---
        def median_survival(risk, stype):
            bt, h0 = baselines[stype]
            surv = np.exp(-h0 * risk)
            idx  = np.searchsorted(-surv, -0.5)
            if idx == 0:
                return bt[0]
            if idx >= len(bt):
                h_last = h0[-1] * risk
                return bt[-1] * (np.log(2) / h_last) if h_last > 0 else bt[-1]
            t0, t1 = bt[idx - 1], bt[idx]
            s0, s1 = surv[idx - 1], surv[idx]
            frac   = (s0 - 0.5) / (s0 - s1) if s0 != s1 else 0.0
            return t0 + frac * (t1 - t0)

        pred_medians = np.array([median_survival(r, s) for r, s in zip(event_risks, event_types)])
        mae = np.mean(np.abs(event_times - pred_medians))

        return {"val_loss": val_loss, "c_index": c_idx, "ibs": ibs, "d_cal": d_cal, "mae": mae}

    def train(self):
        print(f"\nTraining STG-SurviNet for {self.epochs} epochs  [Ablation: {self.ablation_mode}]")

        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")

            train_loss = self.train_epoch()
            self._fit_breslow_on_cal()
            metrics    = self.evaluate(self.val_loader)
            c_index    = metrics["c_index"]

            print(f"  Train Loss : {train_loss:.4f}")
            print(f"  Val Loss   : {metrics['val_loss']:.4f}")
            print(f"  C-Index    : {c_index:.4f}")
            print(f"  IBS        : {metrics['ibs']:.4f}")
            print(f"  D-Cal stat : {metrics['d_cal']['statistic']:.4f}")
            print(f"  D-Cal p    : {metrics['d_cal']['p_value']:.4f}")
            print(f"  MAE        : {metrics['mae']:.2f} days")
            print(f"  LR         : {self.optimizer.param_groups[2]['lr']:.6f}")

            if c_index > self.best_c_index:
                self.best_c_index     = c_index
                self.patience_counter = 0
                self.save_checkpoint()
                print(f"  --> New best C-Index {c_index:.4f} — checkpoint saved.")
            else:
                self.patience_counter += 1
                print(f"  No improvement ({self.patience_counter}/{self.patience})")

            self.scheduler.step(c_index)

            if self.patience_counter >= self.patience:
                print(f"\nEarly stopping after {epoch + 1} epochs.")
                break

    def test(self):
        print("\n" + "=" * 60)
        print("Evaluating on test set...")
        self.load_checkpoint()
        self._fit_breslow_on_cal()
        metrics = self.evaluate(self.test_loader)

        print(f"  Val Loss   : {metrics['val_loss']:.4f}")
        print(f"  C-Index    : {metrics['c_index']:.4f}")
        print(f"  IBS        : {metrics['ibs']:.4f}")
        print(f"  D-Cal stat : {metrics['d_cal']['statistic']:.4f}")
        print(f"  D-Cal p    : {metrics['d_cal']['p_value']:.4f}")
        print(f"  MAE        : {metrics['mae']:.2f} days")
        return metrics

    def save_checkpoint(self):
        os.makedirs(self.train_cfg["checkpoint_dir"], exist_ok=True)
        path = os.path.join(self.train_cfg["checkpoint_dir"], self.train_cfg["checkpoint_name"])
        torch.save(self.model.state_dict(), path)

    def load_checkpoint(self):
        path = os.path.join(self.train_cfg["checkpoint_dir"], self.train_cfg["checkpoint_name"])
        if not os.path.exists(path):
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        self.model.load_state_dict(torch.load(path, map_location=self.device))

## Dataset

In [20]:
N_NODES = 77
TEMPORAL_WINDOW = 60

class STGSurviNetDataset(Dataset):
    def __init__(self, df: pd.DataFrame, node_feature_df: pd.DataFrame = None):
        self.df = df.copy()

        grid_categories = self.df["electricity_grid"].astype("category").cat.codes  # -1 for NaN automatically
        self.df["electricity_grid_enc"] = grid_categories

        feat_df = node_feature_df.copy() if node_feature_df is not None else self.df

        self.sr_types = feat_df['sr_type'].unique()
        for cat in self.sr_types:
            self.df[f"_is_{cat}"] = (self.df["sr_type"] == cat).astype(float)

        self._build_spatial(feat_df)
        self._build_temporal(feat_df)
        self._build_incident_features()

        self.node_indices = torch.tensor(self.df["community_area"].values - 1, dtype=torch.long)
        self.x_spatial  = torch.tensor(self.x_spatial,  dtype=torch.float32)
        self.x_temporal = torch.tensor(self.x_temporal, dtype=torch.float32)
        self.durations  = torch.tensor(self.df['duration_days'].values, dtype=torch.float32)
        self.events     = torch.tensor(self.df['event'].values, dtype=torch.float32)
        self.sr_type_map = {s: i for i, s in enumerate(sorted(self.sr_types))}
        self.sr_type_enc = torch.tensor(
            self.df['sr_type'].map(self.sr_type_map).values, dtype=torch.long
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "node_idx":          self.node_indices[idx],
            "incident_features": torch.tensor(self.incident_matrix[idx], dtype=torch.float32),
            "duration_days":     self.durations[idx],
            "event":             self.events[idx],
            "sr_type_enc":       self.sr_type_enc[idx],
        }

    def _build_spatial(self, feat_df):
        features = []
        for cat in self.sr_types:
            feat_df[f"_is_{cat}"] = (feat_df["sr_type"] == cat).astype(float)

        grouped = feat_df.groupby("community_area")
        for area_id in range(1, N_NODES + 1):
            g = grouped.get_group(area_id) if area_id in grouped.groups else pd.DataFrame()
            if g.empty:
                features.append(np.zeros(8 + len(self.sr_types)))
                continue

            total = len(g)
            type_props    = [g[f"_is_{cat}"].sum() / total for cat in self.sr_types]
            lat_range     = g["latitude"].max()  - g["latitude"].min()  + 1e-6
            lon_range     = g["longitude"].max() - g["longitude"].min() + 1e-6
            density       = total / (lat_range * lon_range * 9435)
            mean_duration = g.loc[g["event"] == 1, "duration_days"].mean() or 0.0
            ward_conc     = g["ward"].value_counts(normalize=True).mean()
            ed_conc       = g["electrical_district"].value_counts(normalize=True).mean()
            grid_conc     = g["electricity_grid"].value_counts(normalize=True).mean()
            hour_mean     = g["created_hour"].mean()
            dow_mean      = g["created_day_of_week"].mean()
            month_mean    = g["created_month"].mean()

            features.append(np.array([
                *type_props, density, mean_duration,
                ward_conc, ed_conc, grid_conc,
                hour_mean, dow_mean, month_mean,
            ], dtype=np.float32))

        self.x_spatial = np.stack(features, axis=0)

    def _build_temporal(self, feat_df):
        start_date = pd.to_datetime(feat_df["created_date"]).max().normalize() - pd.Timedelta(days=TEMPORAL_WINDOW - 1)
        date_range = pd.date_range(start=start_date, periods=TEMPORAL_WINDOW, freq="D")
        sequences  = np.zeros((N_NODES, 1, TEMPORAL_WINDOW), dtype=np.float32)

        window_df = feat_df[pd.to_datetime(feat_df["created_date"]) >= start_date].copy()
        window_df["date"] = pd.to_datetime(window_df["created_date"]).dt.normalize()

        for area_id in range(1, N_NODES + 1):
            g      = window_df[window_df["community_area"] == area_id]
            counts = g.groupby("date").size().reindex(date_range, fill_value=0)
            sequences[area_id - 1, 0, :] = counts.values.astype(np.float32)

        self.x_temporal = sequences

    def _build_incident_features(self):
        sr_type_cols  = [f"_is_{cat}" for cat in self.sr_types]
        sr_type_feats = self.df[sr_type_cols].values.astype(np.float32)

        ward_vals = self.df["ward"].fillna(-1).values.astype(np.float32)
        ed_vals   = self.df["electrical_district"].fillna(-1).values.astype(np.float32)

        month = self.df["created_month"].values.astype(np.float32)
        hour  = self.df["created_hour"].values.astype(np.float32)
        dow   = self.df["created_day_of_week"].values.astype(np.float32)

        created = pd.to_datetime(self.df["created_date"])
        backlog = np.zeros(len(self.df), dtype=np.float32)

        temp_df = pd.DataFrame({'date': created, 'idx': np.arange(len(self.df))})

        # Loop exactly 77 times instead of 540,000 times
        for area in self.df["community_area"].unique():
            mask = self.df["community_area"] == area
            area_df = temp_df[mask].sort_values('date')

            # Apply 60-day rolling time window purely on this area's subset
            area_df = area_df.set_index('date')
            counts = area_df['idx'].rolling('60d').count().values - 1

            # Scatter the calculated counts back to their original row index
            backlog[area_df['idx'].values] = counts

        individual_feats = np.stack([
            np.sin(2 * np.pi * month / 12),
            np.cos(2 * np.pi * month / 12),
            np.sin(2 * np.pi * hour / 24),
            np.cos(2 * np.pi * hour / 24),
            np.sin(2 * np.pi * dow / 7),
            np.cos(2 * np.pi * dow / 7),
            self.df["latitude"].values.astype(np.float32),
            self.df["longitude"].values.astype(np.float32),
            ward_vals,
            ed_vals,
            self.df["electricity_grid_enc"].values.astype(np.float32),
            backlog,
        ], axis=1)

        self.incident_matrix = np.concatenate([sr_type_feats, individual_feats], axis=1)

In [21]:
def get_loaders(df, config):
    splits = config["data"]["splits"]

    indices = np.random.permutation(len(df))
    n_tr  = int(len(df) * splits[0])
    n_val = int(len(df) * splits[1])
    n_cal = int(len(df) * splits[2])

    tr_idx  = indices[:n_tr]
    val_idx = indices[n_tr : n_tr + n_val]
    cal_idx = indices[n_tr + n_val : n_tr + n_val + n_cal]
    te_idx  = indices[n_tr + n_val + n_cal :]

    train_df = df.iloc[tr_idx].reset_index(drop=True)
    dataset  = STGSurviNetDataset(df.reset_index(drop=True), node_feature_df=train_df)

    train_node_ids = dataset.node_indices[tr_idx].unique()

    # Scale spatial
    mean = dataset.x_spatial[train_node_ids].mean(dim=0)
    std  = dataset.x_spatial[train_node_ids].std(dim=0) + 1e-8
    dataset.x_spatial = (dataset.x_spatial - mean) / std

    # Scale temporal
    t_mean = dataset.x_temporal[train_node_ids].mean()
    t_std  = dataset.x_temporal[train_node_ids].std() + 1e-8
    dataset.x_temporal = (dataset.x_temporal - t_mean) / t_std

    # Scale incident features
    inc      = dataset.incident_matrix
    inc_mean = inc[tr_idx].mean(axis=0)
    inc_std  = inc[tr_idx].std(axis=0) + 1e-8
    dataset.incident_matrix = (inc - inc_mean) / inc_std

    bs = config["data"]["batch_size"]
    train_loader = DataLoader(Subset(dataset, tr_idx),  batch_size=bs, shuffle=True)
    val_loader   = DataLoader(Subset(dataset, val_idx), batch_size=bs, shuffle=False)
    cal_loader   = DataLoader(Subset(dataset, cal_idx), batch_size=bs, shuffle=False)
    test_loader  = DataLoader(Subset(dataset, te_idx),  batch_size=bs, shuffle=False)

    return train_loader, val_loader, cal_loader, test_loader, dataset.x_spatial, dataset.x_temporal

# Comparasion

In [22]:
train_loader, val_loader, cal_loader, test_loader, full_x_spatial, full_x_temporal = get_loaders(df_clean, config)

## STGSurviNet (No Ablation)

In [23]:
# Train the full model only when its checkpoint is not available
from pathlib import Path

configured_checkpoint_dir = Path(config["training"]["checkpoint_dir"])
downloads_checkpoint_dir = Path.home() / "Downloads" / "checkpoints"
checkpoint_search = [
    configured_checkpoint_dir / "best.pt",
    downloads_checkpoint_dir / "best.pt",
    *list((Path.home() / "Downloads").glob("*/checkpoints/best.pt")),
]
existing_full_checkpoint = next(
    (path for path in checkpoint_search if path.exists()),
    None,
)
if existing_full_checkpoint is not None:
    config["training"]["checkpoint_dir"] = str(
        existing_full_checkpoint.parent
    )

trainer = Trainer(
    STGSurviNet(
        spatial_in=config["model"]["spatial_in"],
        temporal_in=config["model"]["temporal_in"],
        gcn_out=config["model"]["gcn_out"],
        tcn_out=config["model"]["tcn_out"],
        n_incident_features=config["model"]["n_incident_features"],
    ),
    config,
    train_loader,
    val_loader,
    cal_loader,
    test_loader,
    edge_index,
    edge_weight,
    full_x_spatial,
    full_x_temporal,
)

full_checkpoint = os.path.join(
    config["training"]["checkpoint_dir"],
    config["training"]["checkpoint_name"],
)
if os.path.exists(full_checkpoint):
    print(f"Using existing checkpoint: {full_checkpoint}")
else:
    trainer.train()

no_ablation_res = trainer.test()



Training STG-SurviNet for 50 epochs  [Ablation: None]

Epoch 1/50


Evaluating: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]


  Train Loss : 9.3746
  Val Loss   : 9.1060
  C-Index    : 0.6542
  IBS        : 0.1463
  D-Cal stat : 22131.3044
  D-Cal p    : 0.0000
  MAE        : 15.42 days
  LR         : 0.001000
  --> New best C-Index 0.6542 — checkpoint saved.

Epoch 2/50


Evaluating: 100%|██████████| 4/4 [00:04<00:00,  1.05s/it]


  Train Loss : 9.3307
  Val Loss   : 9.0642
  C-Index    : 0.6768
  IBS        : 0.1385
  D-Cal stat : 18669.2722
  D-Cal p    : 0.0000
  MAE        : 15.16 days
  LR         : 0.001000
  --> New best C-Index 0.6768 — checkpoint saved.

Epoch 3/50


...

  Train Loss : 9.2826
  Val Loss   : 8.9877
  C-Index    : 0.7051
  IBS        : 0.1209
  D-Cal stat : 16070.5679
  D-Cal p    : 0.0000
  MAE        : 13.94 days
  LR         : 0.000063
  --> New best C-Index 0.7051 — checkpoint saved.

Epoch 50/50


Evaluating: 100%|██████████| 4/4 [00:03<00:00,  1.04it/s]


  Train Loss : 9.2826
  Val Loss   : 8.9877
  C-Index    : 0.7050
  IBS        : 0.1209
  D-Cal stat : 16057.3775
  D-Cal p    : 0.0000
  MAE        : 13.94 days
  LR         : 0.000063
  No improvement (1/15)

Evaluating on test set...


Evaluating: 100%|██████████| 2/2 [00:01<00:00,  1.37it/s]


  Val Loss   : 9.0751
  C-Index    : 0.7075
  IBS        : 0.1209
  D-Cal stat : 8633.8229
  D-Cal p    : 0.0000
  MAE        : 13.68 days


## STGSurviNet (Spatial Ablation)

In [ ]:
# Use an independent configuration so the main checkpoint name is unchanged
from copy import deepcopy

config_spat = deepcopy(config)
config_spat["ablation_mode"] = "no_spatial"
config_spat["training"]["checkpoint_name"] = "best_no_spat.pt"

trainer = Trainer(
    STGSurviNet(
        spatial_in=config_spat["model"]["spatial_in"],
        temporal_in=config_spat["model"]["temporal_in"],
        gcn_out=config_spat["model"]["gcn_out"],
        tcn_out=config_spat["model"]["tcn_out"],
        n_incident_features=config_spat["model"]["n_incident_features"],
    ),
    config_spat,
    train_loader,
    val_loader,
    cal_loader,
    test_loader,
    edge_index,
    edge_weight,
    full_x_spatial,
    full_x_temporal,
)

spatial_checkpoint = os.path.join(
    config_spat["training"]["checkpoint_dir"],
    config_spat["training"]["checkpoint_name"],
)
if os.path.exists(spatial_checkpoint):
    print(f"Using existing checkpoint: {spatial_checkpoint}")
else:
    trainer.train()

spatial_ablation_res = trainer.test()


## STGSurviNet (Temporal Ablation)

In [ ]:
# Use an independent configuration so other checkpoint names are unchanged
from copy import deepcopy

config_temp = deepcopy(config)
config_temp["ablation_mode"] = "no_temporal"
config_temp["training"]["checkpoint_name"] = "best_no_temp.pt"

trainer = Trainer(
    STGSurviNet(
        spatial_in=config_temp["model"]["spatial_in"],
        temporal_in=config_temp["model"]["temporal_in"],
        gcn_out=config_temp["model"]["gcn_out"],
        tcn_out=config_temp["model"]["tcn_out"],
        n_incident_features=config_temp["model"]["n_incident_features"],
    ),
    config_temp,
    train_loader,
    val_loader,
    cal_loader,
    test_loader,
    edge_index,
    edge_weight,
    full_x_spatial,
    full_x_temporal,
)

temporal_checkpoint = os.path.join(
    config_temp["training"]["checkpoint_dir"],
    config_temp["training"]["checkpoint_name"],
)
if os.path.exists(temporal_checkpoint):
    print(f"Using existing checkpoint: {temporal_checkpoint}")
else:
    trainer.train()

temporal_ablation_res = trainer.test()


## STGSurviNet (Incident Features Ablation)

In [ ]:
# Use an independent configuration so other checkpoint names are unchanged
from copy import deepcopy

config_inci = deepcopy(config)
config_inci["ablation_mode"] = "no_incident"
config_inci["training"]["checkpoint_name"] = "best_no_inci.pt"

trainer = Trainer(
    STGSurviNet(
        spatial_in=config_inci["model"]["spatial_in"],
        temporal_in=config_inci["model"]["temporal_in"],
        gcn_out=config_inci["model"]["gcn_out"],
        tcn_out=config_inci["model"]["tcn_out"],
        n_incident_features=config_inci["model"]["n_incident_features"],
    ),
    config_inci,
    train_loader,
    val_loader,
    cal_loader,
    test_loader,
    edge_index,
    edge_weight,
    full_x_spatial,
    full_x_temporal,
)

incident_checkpoint = os.path.join(
    config_inci["training"]["checkpoint_dir"],
    config_inci["training"]["checkpoint_name"],
)
if os.path.exists(incident_checkpoint):
    print(f"Using existing checkpoint: {incident_checkpoint}")
else:
    trainer.train()

incident_ablation_res = trainer.test()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Prepare Data
results_data = {
    "Baseline (None)": no_ablation_res,
    "No Spatial": spatial_ablation_res,
    "No Temporal": temporal_ablation_res,
    "No Incident Feature": incident_ablation_res
}
df_plot = pd.DataFrame(results_data).T.reset_index()
df_plot.columns = ['Ablation', 'Val Loss', 'C-Index', 'IBS', 'D-Cal', 'MAE']
df_plot['D-Cal P-Value'] = df_plot['D-Cal'].apply(lambda x: x['p_value'])
df_plot['D-Cal Statistic'] = df_plot['D-Cal'].apply(lambda x: x['statistic'])

# Metrics to plot
metrics = ['C-Index', 'IBS', 'MAE', 'D-Cal Statistic', 'D-Cal P-Value']

# 2. Create Subplots (1 row, 5 columns)
fig = make_subplots(rows=1, cols=5, subplot_titles=metrics, shared_yaxes=False)

# 3. Add traces for each metric
colors = px.colors.qualitative.Bold
for i, metric in enumerate(metrics, 1):
    for j, ablation in enumerate(df_plot['Ablation']):
        fig.add_trace(
            go.Bar(
                name=ablation,
                x=[ablation],
                y=[df_plot.loc[df_plot['Ablation'] == ablation, metric].values[0]],
                marker_color=colors[j],
                showlegend=(i == 1), # Only show legend once
                texttemplate='%{y:.2f}',
                textposition='auto'
            ),
            row=1, col=i
        )

# 4. Final Layout
fig.update_layout(
    height=500,
    width=1400,
    title_text="STG-SurviNet Ablation Study: Metric Performance (Independent Scales)",
    showlegend=True
)
save_plotly_png(fig, "06_stg_survinet_ablation_study.png", width=2000, height=850)
fig.show()

# Classical Model Comparison and Pattern Analysis



## Additional Imports and Analysis Parameters


In [26]:
# Load survival baseline, evaluation, and clustering tools
from sklearn.impute import SimpleImputer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sksurv.nonparametric import kaplan_meier_estimator
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored, integrated_brier_score
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

analysis_config = {
    "random_state": 42,
    "gmm_k_range": range(2, 9),
    "rsf_n_estimators": 20,
    "rsf_max_depth": 7,
    "rsf_min_samples_split": 1000,
    "rsf_min_samples_leaf": 500,
    "prediction_chunk_size": 512,
    "hazard_inference_batch_size": 8192,
}


## Dataset Split and Feature Matrix Preparation


In [27]:
# Reuse the exact split and fitted scaling created for STG-SurviNet
required_split_objects = [
    "train_loader",
    "val_loader",
    "cal_loader",
    "test_loader",
    "full_x_spatial",
    "full_x_temporal",
]
missing_split_objects = [
    name for name in required_split_objects if name not in globals()
]
if missing_split_objects:
    raise RuntimeError(
        "Run the Dataset loader cell and the STG-SurviNet model cells first. "
        f"Missing objects: {', '.join(missing_split_objects)}"
    )

base_dataset = train_loader.dataset.dataset
loader_datasets = [
    val_loader.dataset.dataset,
    cal_loader.dataset.dataset,
    test_loader.dataset.dataset,
]
if any(dataset is not base_dataset for dataset in loader_datasets):
    raise RuntimeError(
        "The train, validation, calibration, and test loaders "
        "do not share the same dataset."
    )

train_idx = np.asarray(train_loader.dataset.indices, dtype=int)
val_idx = np.asarray(val_loader.dataset.indices, dtype=int)
cal_idx = np.asarray(cal_loader.dataset.indices, dtype=int)
test_idx = np.asarray(test_loader.dataset.indices, dtype=int)

all_split_idx = np.concatenate([
    train_idx,
    val_idx,
    cal_idx,
    test_idx,
])
if len(np.unique(all_split_idx)) != len(all_split_idx):
    raise RuntimeError(
        "Dataset splits overlap. Recreate the loaders before running this section."
    )
if len(all_split_idx) != len(base_dataset):
    raise RuntimeError(
        "Dataset splits do not cover the complete cleaned dataset."
    )

def make_structured_y(indices):
    events = base_dataset.events[indices].cpu().numpy().astype(bool)
    times = base_dataset.durations[indices].cpu().numpy().astype(float)
    return np.array(
        list(zip(events, times)),
        dtype=[("event", bool), ("time", float)],
    )

node_idx_all = base_dataset.node_indices.cpu().numpy()
incident_all = np.asarray(
    base_dataset.incident_matrix,
    dtype=np.float32,
)
spatial_by_incident = (
    base_dataset.x_spatial[node_idx_all]
    .cpu()
    .numpy()
    .astype(np.float32)
)
temporal_by_incident = (
    base_dataset.x_temporal[node_idx_all]
    .cpu()
    .numpy()
    .astype(np.float32)
    .squeeze(1)
)

temporal_summary = np.column_stack([
    temporal_by_incident.mean(axis=1),
    temporal_by_incident.std(axis=1),
    temporal_by_incident.max(axis=1),
    temporal_by_incident[:, -1],
    temporal_by_incident.sum(axis=1),
    temporal_by_incident[:, -1] - temporal_by_incident[:, 0],
]).astype(np.float32)

X_traditional = np.concatenate(
    [incident_all, spatial_by_incident, temporal_summary],
    axis=1,
)
if not np.isfinite(X_traditional).all():
    raise ValueError(
        "The classical feature matrix contains non-finite values."
    )

y_train = make_structured_y(train_idx)
y_test = make_structured_y(test_idx)

# Important: no sampling, both classical models use every STG training record
cox_fit_idx = train_idx.copy()
rsf_fit_idx = train_idx.copy()
metric_idx = test_idx.copy()

y_cox_fit = make_structured_y(cox_fit_idx)
y_rsf_fit = make_structured_y(rsf_fit_idx)
y_metric = make_structured_y(metric_idx)
X_metric = X_traditional[metric_idx]

baseline_feature_summary = pd.DataFrame({
    "split": [
        "train",
        "validation",
        "calibration",
        "test",
        "metric_test",
    ],
    "n_records": [
        len(train_idx),
        len(val_idx),
        len(cal_idx),
        len(test_idx),
        len(metric_idx),
    ],
    "event_rate": [
        float(base_dataset.events[train_idx].mean()),
        float(base_dataset.events[val_idx].mean()),
        float(base_dataset.events[cal_idx].mean()),
        float(base_dataset.events[test_idx].mean()),
        float(base_dataset.events[metric_idx].mean()),
    ],
})
baseline_feature_summary


# Verify that every comparison model uses the exact STG-SurviNet split
import hashlib

stg_train_records = len(train_loader.dataset)
stg_test_records = len(test_loader.dataset)

if not np.array_equal(cox_fit_idx, train_idx):
    raise RuntimeError("Cox PH training indices differ from STG-SurviNet.")
if not np.array_equal(rsf_fit_idx, train_idx):
    raise RuntimeError("RSF training indices differ from STG-SurviNet.")
if not np.array_equal(metric_idx, test_idx):
    raise RuntimeError("Classical evaluation indices differ from STG-SurviNet.")
if len(y_cox_fit) != stg_train_records:
    raise RuntimeError("Cox PH does not use every STG-SurviNet training record.")
if len(y_rsf_fit) != stg_train_records:
    raise RuntimeError("RSF does not use every STG-SurviNet training record.")
if len(y_metric) != stg_test_records:
    raise RuntimeError("Classical models do not use every STG-SurviNet test record.")

classical_split_signature = hashlib.sha256(
    train_idx.tobytes() + test_idx.tobytes()
).hexdigest()

comparison_data_audit = pd.DataFrame({
    "model": [
        "STG-SurviNet",
        "Kaplan-Meier",
        "Cox PH",
        "Random Survival Forest",
    ],
    "training_records": [
        stg_train_records,
        len(y_train),
        len(y_cox_fit),
        len(y_rsf_fit),
    ],
    "test_records": [
        stg_test_records,
        len(y_metric),
        len(y_metric),
        len(y_metric),
    ],
    "training_source": [
        "train_idx",
        "train_idx",
        "train_idx",
        "train_idx",
    ],
    "evaluation_source": [
        "test_idx",
        "test_idx",
        "test_idx",
        "test_idx",
    ],
})
comparison_data_audit


,model,training_records,test_records,training_source,evaluation_source
0,STG-SurviNet,324975,54163,train_idx,test_idx
1,Kaplan-Meier,324975,54163,train_idx,test_idx
2,Cox PH,324975,54163,train_idx,test_idx
3,Random Survival Forest,324975,54163,train_idx,test_idx


## Survival Metric Helper Functions


In [28]:
def safe_time_grid(y_train_ref, y_eval_ref, n_times=10):
    observed_eval = y_eval_ref["time"][y_eval_ref["event"]]
    if len(observed_eval) == 0:
        raise ValueError("The evaluation split contains no completed events.")

    lower = max(
        float(np.min(y_train_ref["time"])) + 1e-6,
        float(np.min(y_eval_ref["time"])) + 1e-6,
    )
    upper = min(
        float(np.max(y_train_ref["time"])) - 1e-6,
        float(np.max(y_eval_ref["time"])) - 1e-6,
    )
    if lower >= upper:
        raise ValueError("Training and test follow-up ranges do not overlap.")

    grid = np.unique(
        np.percentile(observed_eval, np.linspace(10, 90, n_times))
    )
    grid = grid[(grid > lower) & (grid < upper)]
    if len(grid) < 2:
        grid = np.linspace(lower, upper, n_times)
    return grid

def evaluate_step_function(survival_function, time_value):
    # Extend the final estimated survival value beyond the fitted time domain
    if time_value < survival_function.x[0]:
        return 1.0
    if time_value > survival_function.x[-1]:
        return float(survival_function.y[-1])
    return float(survival_function(time_value))

def predict_survival_in_chunks(
    model,
    feature_matrix,
    y_eval_ref,
    times_grid,
    chunk_size,
    description,
):
    n_records = len(feature_matrix)
    risk_scores = np.empty(n_records, dtype=np.float64)
    survival_matrix = np.empty(
        (n_records, len(times_grid)),
        dtype=np.float32,
    )
    event_survival_probability = np.full(
        n_records,
        np.nan,
        dtype=np.float64,
    )

    for start in tqdm(
        range(0, n_records, chunk_size),
        desc=description,
    ):
        end = min(start + chunk_size, n_records)
        feature_chunk = feature_matrix[start:end]

        risk_scores[start:end] = model.predict(feature_chunk)
        survival_functions = model.predict_survival_function(feature_chunk)

        for local_index, survival_function in enumerate(survival_functions):
            global_index = start + local_index
            survival_matrix[global_index] = [
                evaluate_step_function(survival_function, time_value)
                for time_value in times_grid
            ]
            if y_eval_ref["event"][global_index]:
                event_survival_probability[global_index] = (
                    evaluate_step_function(
                        survival_function,
                        y_eval_ref["time"][global_index],
                    )
                )

        del survival_functions

    return risk_scores, survival_matrix, event_survival_probability

def d_calibration_from_probabilities(probabilities, n_bins=10):
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities[np.isfinite(probabilities)]
    if len(probabilities) == 0:
        raise ValueError("D-calibration requires completed events.")

    probabilities = np.clip(probabilities, 0.0, 1.0)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    counts, _ = np.histogram(probabilities, bins=bin_edges)
    expected = len(probabilities) / n_bins
    statistic = np.sum((counts - expected) ** 2 / expected)
    p_value = 1.0 - chi2.cdf(statistic, df=n_bins - 1)
    return {
        "statistic": float(statistic),
        "p_value": float(p_value),
        "counts": counts,
    }

def median_from_survival_matrix(survival_matrix, times_grid):
    medians = np.full(
        survival_matrix.shape[0],
        times_grid[-1],
        dtype=float,
    )
    for row_index, row in enumerate(survival_matrix):
        below_half = np.where(row <= 0.5)[0]
        if len(below_half) == 0:
            continue

        crossing_index = below_half[0]
        if crossing_index == 0:
            medians[row_index] = times_grid[0]
            continue

        s0, s1 = row[crossing_index - 1], row[crossing_index]
        t0, t1 = times_grid[crossing_index - 1], times_grid[crossing_index]
        fraction = (s0 - 0.5) / (s0 - s1) if s0 != s1 else 0.0
        medians[row_index] = t0 + fraction * (t1 - t0)
    return medians

def evaluate_survival_predictions(
    model_name,
    risk_scores,
    survival_matrix,
    event_survival_probability,
    y_train_ref,
    y_eval_ref,
    times_grid,
):
    risk_scores = np.asarray(risk_scores, dtype=float)
    survival_matrix = np.asarray(survival_matrix, dtype=float)

    if len(risk_scores) != len(y_eval_ref):
        raise ValueError(
            f"{model_name}: prediction count does not match test data."
        )
    if survival_matrix.shape != (len(y_eval_ref), len(times_grid)):
        raise ValueError(
            f"{model_name}: survival matrix has an invalid shape."
        )
    if not np.isfinite(risk_scores).all():
        raise ValueError(f"{model_name}: risk predictions are not finite.")
    if not np.isfinite(survival_matrix).all():
        raise ValueError(
            f"{model_name}: survival predictions are not finite."
        )

    c_index = concordance_index_censored(
        y_eval_ref["event"],
        y_eval_ref["time"],
        risk_scores,
    )[0]
    ibs = integrated_brier_score(
        y_train_ref,
        y_eval_ref,
        np.clip(survival_matrix, 0.0, 1.0),
        times_grid,
    )

    event_mask = y_eval_ref["event"]
    d_cal = d_calibration_from_probabilities(
        event_survival_probability[event_mask]
    )
    predicted_median = median_from_survival_matrix(
        survival_matrix[event_mask],
        times_grid,
    )
    mae = np.mean(
        np.abs(y_eval_ref["time"][event_mask] - predicted_median)
    )

    return {
        "model": model_name,
        "c_index": float(c_index),
        "ibs": float(ibs),
        "d_cal_statistic": float(d_cal["statistic"]),
        "d_cal_p_value": float(d_cal["p_value"]),
        "mae": float(mae),
    }

times_grid = safe_time_grid(y_train, y_metric)
times_grid


array([ 1.,  2.,  4.,  6.,  9., 17., 43.])

## Classical Survival Baseline Comparison


In [29]:
# Compare all survival models using identical train and test records
required_comparison_objects = [
    "X_traditional",
    "X_metric",
    "y_train",
    "y_cox_fit",
    "y_rsf_fit",
    "y_metric",
    "cox_fit_idx",
    "rsf_fit_idx",
    "times_grid",
    "evaluate_survival_predictions",
    "predict_survival_in_chunks",
]
missing_comparison_objects = [
    name for name in required_comparison_objects if name not in globals()
]
if missing_comparison_objects:
    raise RuntimeError(
        "Run Additional Imports, Dataset Split Preparation, and "
        "Survival Metric Helper Functions first. Missing objects: "
        + ", ".join(missing_comparison_objects)
    )

baseline_results = []
prediction_chunk_size = analysis_config["prediction_chunk_size"]

# Kaplan-Meier population reference
km_time, km_survival = kaplan_meier_estimator(
    y_train["event"],
    y_train["time"],
)

def km_survival_function(time_value):
    time_index = np.searchsorted(km_time, time_value, side="right") - 1
    return 1.0 if time_index < 0 else float(km_survival[time_index])

km_risk = np.zeros(len(y_metric), dtype=float)
km_survival_matrix = np.tile(
    [km_survival_function(time_value) for time_value in times_grid],
    (len(y_metric), 1),
).astype(np.float32)
km_event_probability = np.full(len(y_metric), np.nan, dtype=float)
km_event_indices = np.flatnonzero(y_metric["event"])
km_event_probability[km_event_indices] = [
    km_survival_function(y_metric["time"][index])
    for index in km_event_indices
]
baseline_results.append(
    evaluate_survival_predictions(
        "Kaplan-Meier",
        km_risk,
        km_survival_matrix,
        km_event_probability,
        y_train,
        y_metric,
        times_grid,
    )
)

# Cox PH uses all training records
cox_model_is_current = (
    "cox_ph" in globals()
    and hasattr(cox_ph, "coef_")
    and globals().get("cox_split_signature") == classical_split_signature
)
if not cox_model_is_current:
    cox_imputer = SimpleImputer(strategy="median")
    cox_scaler = StandardScaler()
    X_cox_fit = cox_imputer.fit_transform(X_traditional[cox_fit_idx])
    X_cox_fit = cox_scaler.fit_transform(X_cox_fit)
    X_cox_metric = cox_scaler.transform(
        cox_imputer.transform(X_metric)
    )

    cox_ph = CoxPHSurvivalAnalysis(
        alpha=1e-4,
        n_iter=200,
        tol=1e-7,
    )
    cox_ph.fit(X_cox_fit, y_cox_fit)
    cox_split_signature = classical_split_signature
else:
    print("Reusing the fitted Cox PH model from the active kernel.")
    if "X_cox_metric" not in globals():
        X_cox_metric = cox_scaler.transform(
            cox_imputer.transform(X_metric)
        )

cox_risk, cox_survival_matrix, cox_event_probability = (
    predict_survival_in_chunks(
        cox_ph,
        X_cox_metric,
        y_metric,
        times_grid,
        prediction_chunk_size,
        "Cox PH prediction",
    )
)
baseline_results.append(
    evaluate_survival_predictions(
        "Cox PH",
        cox_risk,
        cox_survival_matrix,
        cox_event_probability,
        y_train,
        y_metric,
        times_grid,
    )
)

# Release Cox training arrays before constructing the forest
import gc

if "X_cox_fit" in globals():
    del X_cox_fit
gc.collect()

# RSF uses all training records, one Windows worker, and memory-safe prediction
rsf_model_is_current = (
    "rsf" in globals()
    and hasattr(rsf, "estimators_")
    and globals().get("rsf_split_signature") == classical_split_signature
)
if not rsf_model_is_current:
    rsf_imputer = SimpleImputer(strategy="median")
    X_rsf_fit = rsf_imputer.fit_transform(X_traditional[rsf_fit_idx])
    X_rsf_metric = rsf_imputer.transform(X_metric)

    rsf = RandomSurvivalForest(
        n_estimators=analysis_config["rsf_n_estimators"],
        max_depth=analysis_config["rsf_max_depth"],
        min_samples_split=analysis_config["rsf_min_samples_split"],
        min_samples_leaf=analysis_config["rsf_min_samples_leaf"],
        max_features="sqrt",
        n_jobs=1,
        random_state=analysis_config["random_state"],
        low_memory=False,
    )
    rsf.fit(X_rsf_fit, y_rsf_fit)
    rsf_split_signature = classical_split_signature
else:
    print("Reusing the fitted Random Survival Forest from the active kernel.")
    if "X_rsf_metric" not in globals():
        X_rsf_metric = rsf_imputer.transform(X_metric)

# Important: avoid WinError 1450 from parallel prediction on Windows
rsf.n_jobs = 1
rsf_risk, rsf_survival_matrix, rsf_event_probability = (
    predict_survival_in_chunks(
        rsf,
        X_rsf_metric,
        y_metric,
        times_grid,
        prediction_chunk_size,
        "Random Survival Forest prediction",
    )
)
baseline_results.append(
    evaluate_survival_predictions(
        "Random Survival Forest",
        rsf_risk,
        rsf_survival_matrix,
        rsf_event_probability,
        y_train,
        y_metric,
        times_grid,
    )
)

stg_rows = []
if "no_ablation_res" in globals():
    stg_rows.append({
        "model": "STG-SurviNet",
        "c_index": no_ablation_res["c_index"],
        "ibs": no_ablation_res["ibs"],
        "d_cal_statistic": no_ablation_res["d_cal"]["statistic"],
        "d_cal_p_value": no_ablation_res["d_cal"]["p_value"],
        "mae": no_ablation_res["mae"],
    })

baseline_comparison = pd.DataFrame(baseline_results + stg_rows)
baseline_comparison = baseline_comparison[[
    "model",
    "c_index",
    "ibs",
    "d_cal_statistic",
    "d_cal_p_value",
    "mae",
]]
baseline_comparison.sort_values(
    "c_index",
    ascending=False,
).reset_index(drop=True)


Random Survival Forest prediction: 100%|██████████| 106/106 [00:15<00:00,  6.95it/s]


,model,c_index,ibs,d_cal_statistic,d_cal_p_value,mae
0,STG-SurviNet,0.707517,0.120928,8633.822857,0.0,13.679111
1,Cox PH,0.652668,0.142082,13767.786117,0.0,15.011993
2,Kaplan-Meier,0.500000,0.157920,27378.471325,0.0,15.522101
3,Random Survival Forest,0.430844,0.128990,13469.724910,0.0,14.372195


In [56]:
import pickle

# --- 1. Export Cox PH Pipeline ---
if "cox_ph" in globals():
    cox_bundle = {
        "imputer": cox_imputer,
        "scaler": cox_scaler,
        "model": cox_ph,
    }
    with open("cox_survival_model.pkl", "wb") as f:
        pickle.dump(cox_bundle, f)
    print("Saved Cox PH bundle to 'cox_survival_model.pkl'")

# --- 2. Export Random Survival Forest Pipeline ---
if "rsf" in globals():
    rsf_bundle = {"imputer": rsf_imputer, "model": rsf}
    with open("rsf_survival_model.pkl", "wb") as f:
        pickle.dump(rsf_bundle, f)
    print("Saved RSF bundle to 'rsf_survival_model.pkl'")

if "km_time" in globals() and "km_survival" in globals():
    km_bundle = {"km_time": km_time, "km_survival": km_survival}

    with open("km_baseline_model.pkl", "wb") as f:
        pickle.dump(km_bundle, f)
    print("Saved Kaplan-Meier baseline data to 'km_baseline_model.pkl'")

Saved Cox PH bundle to 'cox_survival_model.pkl'
Saved RSF bundle to 'rsf_survival_model.pkl'
Saved Kaplan-Meier baseline data to 'km_baseline_model.pkl'


## STG-SurviNet Ablation Summary


In [ ]:
# Summarize component contribution from existing ablation results
ablation_records = {
    "Full STG-SurviNet": no_ablation_res,
    "No Spatial": spatial_ablation_res,
    "No Temporal": temporal_ablation_res,
    "No Incident Features": incident_ablation_res,
}

ablation_summary = []
full_c = no_ablation_res["c_index"]
full_ibs = no_ablation_res["ibs"]
full_mae = no_ablation_res["mae"]

for name, res in ablation_records.items():
    ablation_summary.append({
        "model_variant": name,
        "c_index": res["c_index"],
        "delta_c_index_vs_full": res["c_index"] - full_c,
        "ibs": res["ibs"],
        "delta_ibs_vs_full": res["ibs"] - full_ibs,
        "d_cal_statistic": res["d_cal"]["statistic"],
        "d_cal_p_value": res["d_cal"]["p_value"],
        "mae": res["mae"],
        "delta_mae_vs_full": res["mae"] - full_mae,
    })

ablation_summary_df = pd.DataFrame(ablation_summary).sort_values("c_index", ascending=False)
ablation_summary_df

## Spatial Embedding Clustering with GMM


In [30]:
# Load the trained full model for spatial and temporal pattern analysis
from pathlib import Path

required_pattern_objects = [
    "STGSurviNet",
    "config",
    "full_x_spatial",
    "full_x_temporal",
    "edge_index",
    "gdf",
    "base_dataset",
]
missing_pattern_objects = [
    name for name in required_pattern_objects if name not in globals()
]
if missing_pattern_objects:
    raise RuntimeError(
        "Run preprocessing, graph construction, dataset preparation, and "
        "the full STG-SurviNet result cell first. Missing objects: "
        + ", ".join(missing_pattern_objects)
    )

full_model = STGSurviNet(
    spatial_in=config["model"]["spatial_in"],
    temporal_in=config["model"]["temporal_in"],
    gcn_out=config["model"]["gcn_out"],
    tcn_out=config["model"]["tcn_out"],
    n_incident_features=config["model"]["n_incident_features"],
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_candidates = [
    Path(config["training"]["checkpoint_dir"]) / "best.pt",
    Path.cwd() / config["training"]["checkpoint_dir"] / "best.pt",
    Path.home() / "Downloads" / "checkpoints" / "best.pt",
    *list((Path.home() / "Downloads").glob("*/checkpoints/best.pt")),
]
full_checkpoint_path = next(
    (path for path in checkpoint_candidates if path.exists()),
    None,
)
if full_checkpoint_path is None:
    raise FileNotFoundError(
        "The full-model checkpoint best.pt was not found in the current "
        "folder or Downloads/checkpoints."
    )

full_model.load_state_dict(
    torch.load(full_checkpoint_path, map_location=device)
)
full_model.to(device)
full_model.eval()

edge_index_device = edge_index.to(device)
edge_weight_device = (
    edge_weight.to(device) if edge_weight is not None else None
)
full_x_spatial_device = full_x_spatial.to(device)
full_x_temporal_device = full_x_temporal.to(device)

with torch.no_grad():
    gcn_embeddings = full_model.spatial_extractor(
        full_x_spatial_device,
        edge_index_device,
        edge_weight_device,
    ).cpu().numpy()

area_meta = gdf.copy().reset_index(drop=True)
area_name_col = next(
    (
        column
        for column in [
            "community",
            "community_area_name",
            "area_name",
            "community_area",
        ]
        if column in area_meta.columns
    ),
    None,
)
if "area_numbe" not in area_meta.columns:
    raise KeyError("The boundary file has no area_numbe column.")
if len(area_meta) != gcn_embeddings.shape[0]:
    raise ValueError(
        "Boundary rows and GCN node embeddings have different lengths."
    )

best_spatial_gmm = None
spatial_gmm_rows = []
for cluster_count in analysis_config["gmm_k_range"]:
    gmm = GaussianMixture(
        n_components=cluster_count,
        covariance_type="full",
        random_state=analysis_config["random_state"],
    )
    labels = gmm.fit_predict(gcn_embeddings)
    silhouette = (
        silhouette_score(gcn_embeddings, labels)
        if len(np.unique(labels)) > 1
        else np.nan
    )
    current_result = {
        "k": cluster_count,
        "bic": gmm.bic(gcn_embeddings),
        "silhouette": silhouette,
    }
    spatial_gmm_rows.append(current_result)
    if best_spatial_gmm is None or current_result["bic"] < best_spatial_gmm["bic"]:
        best_spatial_gmm = {
            "k": cluster_count,
            "bic": current_result["bic"],
            "model": gmm,
            "labels": labels,
        }

spatial_cluster_df = pd.DataFrame({
    "community_area": area_meta["area_numbe"].astype(int).values,
    "community_name": (
        area_meta[area_name_col].astype(str).values
        if area_name_col
        else area_meta["area_numbe"].astype(str).values
    ),
    "spatial_cluster": best_spatial_gmm["labels"],
})

spatial_cluster_profile = (
    base_dataset.df
    .merge(spatial_cluster_df, on="community_area", how="left")
    .groupby("spatial_cluster")
    .agg(
        n_records=("duration_days", "size"),
        mean_duration=("duration_days", "mean"),
        median_duration=("duration_days", "median"),
        event_rate=("event", "mean"),
        mean_area=("community_area", "mean"),
    )
    .reset_index()
)

(
    pd.DataFrame(spatial_gmm_rows).sort_values("bic"),
    spatial_cluster_profile.sort_values("mean_duration", ascending=False),
)


(   k           bic  silhouette
 0  2 -24841.060025    0.249878
 1  3 -17819.700362    0.214844
 2  4 -10196.453918    0.225376
 3  5  -2593.077836    0.269852
 4  6   5956.932035    0.292350
 5  7  14084.710827    0.362632
 6  8  22593.320979    0.345207,
    spatial_cluster  n_records  mean_duration  median_duration  event_rate  \
 0                0     389266      19.309192              3.0    0.972351   
 1                1     152359      18.346005              3.0    0.967839   
 
    mean_area  
 0  30.210743  
 1  53.628286  )

## Temporal Workload Pattern Analysis


In [31]:
# Analyze 60-day workload patterns and TCN embeddings
created_dates = pd.to_datetime(base_dataset.df["created_date"])
window_start = created_dates.max().normalize() - pd.Timedelta(days=TEMPORAL_WINDOW - 1)
date_range = pd.date_range(window_start, periods=TEMPORAL_WINDOW, freq="D")

raw_temporal_counts = np.zeros((N_NODES, TEMPORAL_WINDOW), dtype=np.float32)
window_df = base_dataset.df.loc[created_dates >= window_start, ["community_area", "created_date"]].copy()
window_df["date"] = pd.to_datetime(window_df["created_date"]).dt.normalize()

for area_id in range(1, N_NODES + 1):
    counts = window_df.loc[window_df["community_area"] == area_id].groupby("date").size().reindex(date_range, fill_value=0)
    raw_temporal_counts[area_id - 1] = counts.values.astype(np.float32)

with torch.no_grad():
    tcn_embeddings = full_model.temporal_extractor(full_x_temporal.to(device)).cpu().numpy()

best_temporal_gmm = None
temporal_gmm_rows = []
for k in analysis_config["gmm_k_range"]:
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=analysis_config["random_state"])
    labels = gmm.fit_predict(tcn_embeddings)
    sil = silhouette_score(tcn_embeddings, labels) if len(np.unique(labels)) > 1 else np.nan
    temporal_gmm_rows.append({"k": k, "bic": gmm.bic(tcn_embeddings), "silhouette": sil})
    if best_temporal_gmm is None or temporal_gmm_rows[-1]["bic"] < best_temporal_gmm["bic"]:
        best_temporal_gmm = {"k": k, "bic": temporal_gmm_rows[-1]["bic"], "model": gmm, "labels": labels}

temporal_cluster_df = spatial_cluster_df[["community_area", "community_name"]].copy()
temporal_cluster_df["temporal_cluster"] = best_temporal_gmm["labels"]
temporal_cluster_df["total_60d_workload"] = raw_temporal_counts.sum(axis=1)
temporal_cluster_df["mean_daily_workload"] = raw_temporal_counts.mean(axis=1)
temporal_cluster_df["max_daily_workload"] = raw_temporal_counts.max(axis=1)
temporal_cluster_df["workload_trend_last_vs_first"] = raw_temporal_counts[:, -1] - raw_temporal_counts[:, 0]

temporal_cluster_profile = (
    temporal_cluster_df.groupby("temporal_cluster")
    .agg(
        n_areas=("community_area", "size"),
        mean_60d_workload=("total_60d_workload", "mean"),
        mean_daily_workload=("mean_daily_workload", "mean"),
        mean_peak_daily_workload=("max_daily_workload", "mean"),
        mean_trend_last_vs_first=("workload_trend_last_vs_first", "mean"),
    )
    .reset_index()
)

citywide_temporal_profile = pd.DataFrame({"date": date_range, "complaint_count": raw_temporal_counts.sum(axis=0)})

top_temporal_areas = temporal_cluster_df.sort_values("total_60d_workload", ascending=False).head(15)
pd.DataFrame(temporal_gmm_rows).sort_values("bic"), temporal_cluster_profile.sort_values("mean_60d_workload", ascending=False), top_temporal_areas


(   k          bic  silhouette
 1  3 -5044.125210    0.238689
 0  2 -4376.409422    0.228580
 3  5 -3814.658776    0.145345
 2  4 -3517.800982    0.212410
 4  6 -2895.440352    0.128411
 6  8  -292.435191    0.147796
 5  7   -91.806248    0.115876,
    temporal_cluster  n_areas  mean_60d_workload  mean_daily_workload  \
 2                 2       10         326.899994             5.448333   
 0                 0       22         247.590912             4.126515   
 1                 1       45         142.822220             2.380370   
 
    mean_peak_daily_workload  mean_trend_last_vs_first  
 2                 19.600000                  0.200000  
 0                 17.090910                  0.227273  
 1                 12.866667                  1.155556  ,
     community_area   community_name  temporal_cluster  total_60d_workload  \
 24              25           AUSTIN                 2               509.0   
 27              28   NEAR WEST SIDE                 2               493

## Delay-Risk Aggregation by Area and Category


In [32]:
# Aggregate full-model hazard scores into interpretable delay-risk patterns
all_log_hazard = np.zeros(len(base_dataset), dtype=np.float32)
inference_batch_size = analysis_config["hazard_inference_batch_size"]
incident_matrix = np.asarray(
    base_dataset.incident_matrix,
    dtype=np.float32,
)

with torch.no_grad():
    for start in tqdm(
        range(0, len(base_dataset), inference_batch_size),
        desc="STG-SurviNet hazard inference",
    ):
        end = min(start + inference_batch_size, len(base_dataset))
        batch_nodes = base_dataset.node_indices[start:end].to(device)
        batch_incident = torch.as_tensor(
            incident_matrix[start:end],
            dtype=torch.float32,
            device=device,
        )
        log_hazard = full_model(
            full_x_spatial_device,
            edge_index_device,
            edge_weight_device,
            full_x_temporal_device,
            node_indices=batch_nodes,
            incident_features=batch_incident,
            ablation_mode=None,
        ).squeeze(-1)
        all_log_hazard[start:end] = log_hazard.cpu().numpy()

risk_pattern_df = base_dataset.df[[
    "community_area",
    "sr_type",
    "duration_days",
    "event",
    "created_date",
]].copy()
risk_pattern_df["hazard_multiplier"] = np.exp(
    np.clip(all_log_hazard, -10, 10)
)
risk_pattern_df["delay_risk_score"] = -all_log_hazard
risk_pattern_df = risk_pattern_df.merge(
    spatial_cluster_df,
    on="community_area",
    how="left",
)
risk_pattern_df = risk_pattern_df.merge(
    temporal_cluster_df[[
        "community_area",
        "temporal_cluster",
        "total_60d_workload",
        "workload_trend_last_vs_first",
    ]],
    on="community_area",
    how="left",
)

if risk_pattern_df[[
    "spatial_cluster",
    "temporal_cluster",
]].isna().any().any():
    raise ValueError(
        "Some complaint records could not be matched to spatial or temporal clusters."
    )

area_delay_risk = (
    risk_pattern_df
    .groupby([
        "community_area",
        "community_name",
        "spatial_cluster",
        "temporal_cluster",
    ])
    .agg(
        n_records=("duration_days", "size"),
        mean_duration=("duration_days", "mean"),
        median_duration=("duration_days", "median"),
        event_rate=("event", "mean"),
        mean_hazard_multiplier=("hazard_multiplier", "mean"),
        mean_delay_risk=("delay_risk_score", "mean"),
        total_60d_workload=("total_60d_workload", "first"),
        workload_trend_last_vs_first=(
            "workload_trend_last_vs_first",
            "first",
        ),
    )
    .reset_index()
    .sort_values("mean_delay_risk", ascending=False)
)

category_delay_risk = (
    risk_pattern_df
    .groupby("sr_type")
    .agg(
        n_records=("duration_days", "size"),
        mean_duration=("duration_days", "mean"),
        median_duration=("duration_days", "median"),
        event_rate=("event", "mean"),
        mean_hazard_multiplier=("hazard_multiplier", "mean"),
        mean_delay_risk=("delay_risk_score", "mean"),
    )
    .reset_index()
    .sort_values("mean_delay_risk", ascending=False)
)

area_category_delay_risk = (
    risk_pattern_df
    .groupby(["community_area", "community_name", "sr_type"])
    .agg(
        n_records=("duration_days", "size"),
        mean_duration=("duration_days", "mean"),
        median_duration=("duration_days", "median"),
        mean_delay_risk=("delay_risk_score", "mean"),
    )
    .reset_index()
)
area_category_delay_risk = area_category_delay_risk[
    area_category_delay_risk["n_records"] >= 30
].sort_values("mean_delay_risk", ascending=False)

(
    area_delay_risk.head(15),
    category_delay_risk,
    area_category_delay_risk.head(20),
)


STG-SurviNet hazard inference: 100%|██████████| 67/67 [00:03<00:00, 17.61it/s]


(    community_area  community_name  spatial_cluster  temporal_cluster  \
 75            76.0           OHARE                0                 0   
 13            14.0     ALBANY PARK                0                 1   
 17            18.0       MONTCLARE                1                 0   
 15            16.0     IRVING PARK                0                 2   
 53            54.0       RIVERDALE                1                 1   
 22            23.0   HUMBOLDT PARK                0                 1   
 28            29.0  NORTH LAWNDALE                0                 0   
 66            67.0  WEST ENGLEWOOD                0                 0   
 9             10.0    NORWOOD PARK                0                 2   
 8              9.0     EDISON PARK                0                 1   
 41            42.0        WOODLAWN                1                 1   
 11            12.0     FOREST GLEN                0                 1   
 35            36.0         OAKLAND   

## Model Comparison Visualization


In [ ]:
metric_plot_df = baseline_comparison.melt(
    id_vars="model",
    value_vars=["c_index", "ibs", "mae", "d_cal_statistic", "d_cal_p_value"],
    var_name="metric",
    value_name="value",
)
metric_plot_df["metric_label"] = metric_plot_df["metric"].map({
    "c_index": "C-Index",
    "ibs": "Integrated Brier Score",
    "mae": "MAE",
    "d_cal_statistic": "D-Calibration Statistic",
    "d_cal_p_value": "D-Calibration P-Value",
})

fig = px.bar(
    metric_plot_df,
    x="value",
    y="model",
    color="model",
    facet_col="metric_label",
    facet_col_wrap=2,
    orientation="h",
    title="Survival Model Metric Comparison",
    labels={
        "value": "Metric value",
        "model": "Model",
        "metric_label": "Metric",
    },
)
fig.update_xaxes(matches=None, automargin=True)
fig.update_yaxes(automargin=True)
fig.update_layout(
    height=760,
    title_x=0.5,
    showlegend=False,
    margin={"r":30, "t":80, "l":120, "b":40},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("metric_label=", "")))
save_plotly_png(fig, "07_survival_model_metric_comparison.png", width=1800, height=1300)
fig.show()

In [46]:
fig.write_html('survival_model_metric_comparison.html')

## Complaint Category Delay-Risk Visualization

In [ ]:
fig = px.bar(
    category_delay_risk,
    x="mean_delay_risk",
    y="sr_type",
    color="sr_type",
    orientation="h",
    title="Mean Delay-Risk Score by Complaint Category",
    labels={
        "sr_type": "Complaint category",
        "mean_delay_risk": "Mean delay-risk score",
    },
)
fig.update_yaxes(automargin=True)
fig.update_layout(
    height=500,
    title_x=0.5,
    showlegend=False,
    margin={"r":30, "t":70, "l":220, "b":50},
)
save_plotly_png(fig, "08_mean_delay_risk_by_complaint_category.png", width=1800, height=900)
fig.show()

In [48]:
fig.write_html('mean_delay_risk_by_complaint_category.png')

## Workload and Delay-Risk Relationship

In [ ]:
fig = px.scatter(
    area_delay_risk,
    x="total_60d_workload",
    y="mean_delay_risk",
    color="spatial_cluster",
    size="n_records",
    hover_data=["community_area", "community_name", "temporal_cluster", "mean_duration", "median_duration", "event_rate"],
    title="Community Area Delay Risk vs 60-Day Workload",
    labels={
        "total_60d_workload": "Total complaints in last 60 days",
        "mean_delay_risk": "Mean delay-risk score",
        "spatial_cluster": "Spatial GMM cluster",
    },
)
fig.update_layout(
    height=620,
    title_x=0.5,
    margin={"r":30, "t":70, "l":80, "b":70},
)
fig.update_xaxes(automargin=True)
fig.update_yaxes(automargin=True)
save_plotly_png(fig, "09_community_area_delay_risk_vs_workload.png", width=1800, height=1000)
fig.show()

In [50]:
fig.write_html('community_area_delay_risk_vs_workload.html')

## Citywide Temporal Workload Visualization

In [ ]:
fig = px.line(
    citywide_temporal_profile,
    x="date",
    y="complaint_count",
    title="Citywide 60-Day Infrastructure Complaint Workload",
    labels={"complaint_count": "Daily complaint count", "date": "Date"},
)
fig.update_traces(
    hovertemplate="Date=%{x|%b %d, %Y}<br>Daily complaint count=%{y}<extra></extra>"
)
fig.update_xaxes(tickformat="%b %d, %Y", tickangle=45, automargin=True)
fig.update_yaxes(automargin=True)
fig.update_layout(
    height=620,
    title_x=0.5,
    margin={"r":30, "t":70, "l":80, "b":110},
)
save_plotly_png(fig, "10_citywide_60_day_complaint_workload.png", width=1800, height=1000)
fig.show()

In [51]:
fig.write_html('citywide_60_day_complaint_workload.html')

## Temporal Cluster Workload Visualization

In [ ]:
fig = px.bar(
    temporal_cluster_profile.sort_values("mean_60d_workload", ascending=False),
    x="temporal_cluster",
    y="mean_60d_workload",
    color="temporal_cluster",
    title="Average 60-Day Workload by Temporal Cluster",
    labels={
        "temporal_cluster": "Temporal GMM cluster",
        "mean_60d_workload": "Average total workload over 60 days",
    },
)
fig.update_layout(
    height=520,
    title_x=0.5,
    showlegend=False,
    margin={"r":30, "t":70, "l":90, "b":70},
)
fig.update_xaxes(automargin=True)
fig.update_yaxes(automargin=True)
save_plotly_png(fig, "11_average_workload_by_temporal_cluster.png", width=1600, height=900)
fig.show()

In [53]:
fig.write_html('average_workload_by_temporal_cluster.html')

## Spatial Cluster Map

In [ ]:
spatial_cluster_map = spatial_cluster_df.copy()
spatial_cluster_map["community_area"] = spatial_cluster_map["community_area"].astype(str)

cluster_duration_order = (
    spatial_cluster_profile.sort_values("mean_duration")
    .reset_index(drop=True)
    .assign(cluster_rank=lambda x: x.index + 1)
)
cluster_label_map = {}
for _, row in cluster_duration_order.iterrows():
    cluster_label_map[row["spatial_cluster"]] = (
        f"Cluster {int(row['cluster_rank'])}: "
        f"avg duration {row['mean_duration']:.2f} days"
    )

spatial_cluster_map["spatial_cluster_label"] = spatial_cluster_map["spatial_cluster"].map(cluster_label_map)

fig = px.choropleth_map(
    spatial_cluster_map,
    geojson=community_areas_geojson,
    locations="community_area",
    featureidkey="properties.area_numbe",
    color="spatial_cluster_label",
    hover_data=["community_area", "community_name", "spatial_cluster_label"],
    map_style="carto-positron",
    zoom=9.72,
    center={"lat": 41.8372, "lon": -87.6860},
    opacity=0.65,
    title="GMM Spatial Clusters of Chicago Community Areas",
    labels={"spatial_cluster_label": "Spatial cluster"},
)

# Fixed Chicago bounds keep the PNG readable in an infographic
chicago_map_bounds = {
    "west": -87.97,
    "east": -87.50,
    "south": 41.62,
    "north": 42.04,
}

fig.update_layout(
    height=650,
    title_x=0.5,
    legend={
        "title_text": "Spatial cluster profile",
        "orientation": "h",
        "x": 0.5,
        "y": -0.02,
        "xanchor": "center",
        "yanchor": "top",
    },
    margin={"r":20, "t":80, "l":20, "b":20},
)
save_plotly_png(fig, "12_gmm_spatial_clusters_map.png", width=1100, height=1200)
fig.show()

In [54]:
fig.write_html('gmm_spatial_clusters_map.html')

## Community Area Delay-Risk Map

In [ ]:
map_area_delay_risk = area_delay_risk[[
    "community_area",
    "community_name",
    "mean_delay_risk",
    "mean_duration",
    "median_duration",
    "total_60d_workload",
]].copy()
map_area_delay_risk["community_area"] = map_area_delay_risk["community_area"].astype(int)

delay_risk_gdf = gdf.copy()
delay_risk_gdf["area_numbe"] = delay_risk_gdf["area_numbe"].astype(int)
delay_risk_gdf = delay_risk_gdf.merge(
    map_area_delay_risk,
    left_on="area_numbe",
    right_on="community_area",
    how="left",
)

delay_risk_gdf["feature_id"] = delay_risk_gdf["area_numbe"].astype(str)
delay_risk_gdf["map_label"] = delay_risk_gdf["community_name"].fillna(delay_risk_gdf["area_numbe"].astype(str))

delay_risk_plot_gdf = delay_risk_gdf[[
    "feature_id",
    "area_numbe",
    "map_label",
    "mean_delay_risk",
    "mean_duration",
    "median_duration",
    "total_60d_workload",
    "geometry",
]].copy()
delay_risk_plot_df = pd.DataFrame(delay_risk_plot_gdf.drop(columns="geometry"))
delay_risk_geojson = json.loads(delay_risk_plot_gdf.to_crs(epsg=4326).to_json())

fig = px.choropleth_map(
    delay_risk_plot_df,
    geojson=delay_risk_geojson,
    locations="feature_id",
    featureidkey="properties.feature_id",
    color="mean_delay_risk",
    hover_data={
        "feature_id": False,
        "area_numbe": True,
        "map_label": True,
        "mean_delay_risk": ":.4f",
        "mean_duration": ":.2f",
        "median_duration": ":.2f",
        "total_60d_workload": ":.0f",
    },
    color_continuous_scale="YlOrRd",
    map_style="carto-positron",
    zoom=9.72,
    center={"lat": 41.8372, "lon": -87.6860},
    opacity=0.70,
    title="Community Area Delay-Risk Map",
    labels={"mean_delay_risk": "Mean delay-risk score"},
)

# Fixed Chicago bounds keep the PNG readable in an infographic
chicago_map_bounds = {
    "west": -87.97,
    "east": -87.50,
    "south": 41.62,
    "north": 42.04,
}

fig.update_layout(
    height=650,
    title_x=0.5,
    coloraxis_colorbar={
        "title": "Mean delay-risk score",
        "orientation": "h",
        "x": 0.5,
        "y": -0.02,
        "xanchor": "center",
        "yanchor": "top",
        "len": 0.72,
        "thickness": 22,
    },
    margin={"r":20, "t":80, "l":20, "b":20},
)
save_plotly_png(fig, "13_community_area_delay_risk_map.png", width=1050, height=1200)
fig.show()

In [55]:
fig.write_html('community_area_delay_risk_map.html')

In [ ]:
analysis_findings = {
    "best_classical_baseline_by_c_index": pd.DataFrame(baseline_results)
        .sort_values("c_index", ascending=False)
        .iloc[0]["model"],

    "best_overall_model_by_c_index": baseline_comparison
        .sort_values("c_index", ascending=False)
        .iloc[0]["model"],

    "highest_delay_risk_category": category_delay_risk.iloc[0]["sr_type"],

    "highest_delay_risk_area": int(area_delay_risk.iloc[0]["community_area"]),

    "selected_spatial_gmm_k": int(best_spatial_gmm["k"]),

    "selected_temporal_gmm_k": int(best_temporal_gmm["k"]),
}

analysis_report_tables = {
    "baseline_comparison": baseline_comparison.sort_values("c_index", ascending=False),
    "ablation_summary": ablation_summary_df,
    "spatial_cluster_profile": spatial_cluster_profile.sort_values("mean_duration", ascending=False),
    "temporal_cluster_profile": temporal_cluster_profile.sort_values("mean_60d_workload", ascending=False),
    "top_delay_risk_areas": area_delay_risk.head(15),
    "top_delay_risk_area_categories": area_category_delay_risk.head(20),
}

main_findings_table = pd.DataFrame([
    {
        "Finding": "Best classical baseline",
        "Result": analysis_findings["best_classical_baseline_by_c_index"],
        "Interpretation": "Classical survival model with the strongest ranking ability based on C-index."
    },
    {
        "Finding": "Best overall model",
        "Result": analysis_findings["best_overall_model_by_c_index"],
        "Interpretation": "Best model among STG-SurviNet and the classical survival baselines."
    },
    {
        "Finding": "Highest delay-risk category",
        "Result": analysis_findings["highest_delay_risk_category"],
        "Interpretation": "Complaint category with the highest average predicted delay-risk score."
    },
    {
        "Finding": "Highest delay-risk community area",
        "Result": analysis_findings["highest_delay_risk_area"],
        "Interpretation": "Community area with the highest average predicted delay-risk score."
    },
    {
        "Finding": "Selected spatial GMM clusters",
        "Result": analysis_findings["selected_spatial_gmm_k"],
        "Interpretation": "Number of clusters selected from learned GCN spatial embeddings using BIC."
    },
    {
        "Finding": "Selected temporal GMM clusters",
        "Result": analysis_findings["selected_temporal_gmm_k"],
        "Interpretation": "Number of clusters selected from learned TCN workload embeddings using BIC."
    },
])

main_findings_table

# Package visualization PNG files
import shutil

visualization_archive = shutil.make_archive(
    str(VISUALIZATION_DIR),
    "zip",
    root_dir=VISUALIZATION_DIR,
)
print(f"Visualization archive: {Path(visualization_archive).resolve()}")
